# ffmpeg_core

## Example 1: FFmpeg's expression language from Python

In [ ]:
# Example 1: expressions compose into the strings FFmpeg's filter options evaluate
import ffmpeg_core.expressions as e

t = e.Expression("t")
fade = e.clip(t * 2, 0, 10)
choose = e.if_(e.gt(t, 5), 1, 0)
assert str(fade) == "clip(t*2,0,10)", str(fade)
assert str(choose).startswith("if(gt(t,5)"), str(choose)
assert str(e.between(t, 1, 3)) == "between(t,1,3)"
print("Example 1:", fade, "|", choose)

In [ ]:
from ffmpeg_core.info import CodecFlags, get_codecs, get_encoders

codecs = {codec.name: codec for codec in get_codecs()}
encoders = {encoder.name for encoder in get_encoders()}
assert {"mpeg4", "aac", "flac", "pcm_s16le", "libvpx-vp9"} <= encoders, sorted(encoders)[:20]
assert codecs["h264"].flags & CodecFlags.decoding and codecs["h264"].flags & CodecFlags.video
assert "libx264" not in encoders
print("Example 2:", len(codecs), "codecs,", len(encoders), "encoders; e.g.", sorted(encoders)[:5])

## Example 3: probe a file and parse ffprobe's answer into objects

In [ ]:
# Example 3: probe_obj runs ffprobe and returns typed dataclasses
import pathlib, subprocess, tempfile
from ffmpeg_core.ffprobe.probe import probe_obj

wav = pathlib.Path(tempfile.mkdtemp()) / "tone.wav"
subprocess.run(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "sine=frequency=1000:sample_rate=16000:duration=0.5",
                str(wav)], check=True)
result = probe_obj(wav)
stream = result.streams.stream[0]
assert stream.codec_name == "pcm_s16le" and int(stream.sample_rate) == 16000, stream
assert abs(float(result.format.duration) - 0.5) < 0.05, result.format.duration
print("Example 3:", stream.codec_name, stream.sample_rate, result.format.duration)

## Example 4: escaping filter arguments

In [ ]:
# Example 4: escape() protects the characters FFmpeg's filter syntax treats specially
from ffmpeg_core.utils.escaping import convert_kwargs_to_cmd_line_args, escape

assert escape("a:b") == "a\\:b", escape("a:b")
assert escape("it's") == "it\\'s", escape("it's")
assert escape(1.5) == "1.5"
args = convert_kwargs_to_cmd_line_args({"vcodec": "mpeg4", "ss": 2, "map": ["0:v", "0:a"]})
# keys come out sorted, and a list value repeats its flag
assert args == ["-map", "0:v", "-map", "0:a", "-ss", "2", "-vcodec", "mpeg4"], args
print("Example 4:", escape("drawtext=text='x:y'"), args)

## Example 5: the FFmpeg 9 data package is found

In [ ]:
# Example 5: ffmpeg_core finds the per-FFmpeg-version data package (ffmpeg_data_v9) by its name prefix
from ffmpeg_core.common import cache

paths = cache._iter_data_cache_paths()
assert any("ffmpeg_data_v9" in str(path) for path in paths), paths
assert all(path.is_dir() for path in paths)
print("Example 5:", [str(path).split("site-packages/")[-1] for path in paths])